# Train Full-Precision CNN and RCNN Decoders

This notebook trains full-precision (32-bit) CNN and RCNN models across all 12 (d, p) configs.
Results are saved to `results_fullprecision.csv` and serve as:
- Experiment 1 comparison (CNN vs RCNN vs MWPM)
- 32-bit baseline for Phase 5 quantization sweep

In [2]:
import numpy as np
import tensorflow as tf
import os
import csv
from datetime import datetime

In [4]:
print(f'Starting full-precision training at {datetime.now()}')
print(f'TensorFlow version: {tf.__version__}')

# Configuration
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]
ROUNDS = 2
DATA_DIR = './datasets'
OUTPUT_DIR = './results'
SEED = 42

# Training hyperparameters
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
PATIENCE = 10
MAX_EPOCHS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}\n')

Starting full-precision training at 2026-05-29 01:07:26.766770
TensorFlow version: 2.21.0
Data dir: ./datasets
Output dir: ./results



In [5]:
def build_cnn(input_shape, output_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Reshape((int(np.sqrt(input_shape[0])), int(np.sqrt(input_shape[0])), 1)),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(output_shape[0], activation='sigmoid')
    ])
    return model

def build_rcnn(input_shape, output_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Reshape((int(np.sqrt(input_shape[0])), int(np.sqrt(input_shape[0])), 1)),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.RepeatVector(1),
        tf.keras.layers.LSTM(32, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(output_shape[0], activation='sigmoid')
    ])
    return model

print('CNN and RCNN architectures defined')

CNN and RCNN architectures defined


In [6]:
results = []

for config_idx, (d, p) in enumerate(CONFIGS, 1):
    print(f'\n[{config_idx}/{len(CONFIGS)}] d={d}, p={p:.3f}')
    print('='*60)
    
    try:
        # Load dataset
        filename = f'{DATA_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz'
        data = np.load(filename)
        det_evts = data['det_evts'].astype(np.float32)
        flips = data['flips'].astype(np.float32)
        
        # Split: 800k train, 100k val, 100k test
        n_train = 800_000
        n_val = 100_000
        
        X_train = det_evts[:n_train]
        y_train = flips[:n_train]
        X_val = det_evts[n_train:n_train+n_val]
        y_val = flips[n_train:n_train+n_val]
        X_test = det_evts[n_train+n_val:]
        y_test = flips[n_train+n_val:]
        
        input_shape = X_train.shape[1:]
        output_shape = y_train.shape[1:]
        
        print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}\n')
        
        # Train CNN
        print('Training CNN...')
        cnn = build_cnn(input_shape, output_shape)
        cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                    loss='binary_crossentropy',
                    metrics=['accuracy'])
        
        cnn_history = cnn.fit(
            X_train, y_train,
            batch_size=BATCH_SIZE,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
            verbose=1
        )
        
        # Evaluate CNN
        cnn_train_loss, cnn_train_acc = cnn.evaluate(X_train, y_train, verbose=0)
        cnn_test_loss, cnn_test_acc = cnn.evaluate(X_test, y_test, verbose=0)
        cnn_pred = (cnn.predict(X_test, verbose=0) > 0.5).astype(float)
        cnn_errors = (cnn_pred != y_test).sum()
        cnn_p_L = cnn_errors / len(y_test)
        cnn_size_kb = cnn.count_params() * 4 / 1024
        cnn_epochs = len(cnn_history.history['loss'])
        
        print(f'CNN Training data loss: {cnn_train_loss:.6f}, accuracy: {cnn_train_acc:.6f}')
        print(f'CNN Test data loss: {cnn_test_loss:.6f}, accuracy: {cnn_test_acc:.6f}')
        print(f'CNN p_L={cnn_p_L:.6f} (trained {cnn_epochs} epochs, size={cnn_size_kb:.1f} KB)\n')
        
        results.append({
            'distance': d,
            'noise': p,
            'decoder': 'CNN-32bit',
            'p_L': cnn_p_L,
            'model_size_kb': cnn_size_kb,
        })
        
        # Train RCNN
        print('Training RCNN...')
        rcnn = build_rcnn(input_shape, output_shape)
        rcnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                     loss='binary_crossentropy',
                     metrics=['accuracy'])
        
        rcnn_history = rcnn.fit(
            X_train, y_train,
            batch_size=BATCH_SIZE,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
            verbose=1
        )
        
        # Evaluate RCNN
        rcnn_train_loss, rcnn_train_acc = rcnn.evaluate(X_train, y_train, verbose=0)
        rcnn_test_loss, rcnn_test_acc = rcnn.evaluate(X_test, y_test, verbose=0)
        rcnn_pred = (rcnn.predict(X_test, verbose=0) > 0.5).astype(float)
        rcnn_errors = (rcnn_pred != y_test).sum()
        rcnn_p_L = rcnn_errors / len(y_test)
        rcnn_size_kb = rcnn.count_params() * 4 / 1024
        rcnn_epochs = len(rcnn_history.history['loss'])
        
        print(f'RCNN Training data loss: {rcnn_train_loss:.6f}, accuracy: {rcnn_train_acc:.6f}')
        print(f'RCNN Test data loss: {rcnn_test_loss:.6f}, accuracy: {rcnn_test_acc:.6f}')
        print(f'RCNN p_L={rcnn_p_L:.6f} (trained {rcnn_epochs} epochs, size={rcnn_size_kb:.1f} KB)\n')
        
        results.append({
            'distance': d,
            'noise': p,
            'decoder': 'RCNN-32bit',
            'p_L': rcnn_p_L,
            'model_size_kb': rcnn_size_kb,
        })
        
    except Exception as e:
        print(f'✗ Error: {e}')
        continue


[1/12] d=3, p=0.001
Train: (800000, 16), Val: (100000, 16), Test: (100000, 16)

Training CNN...
Epoch 1/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9982 - loss: 0.0098 - val_accuracy: 0.9994 - val_loss: 0.0029
Epoch 2/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9994 - loss: 0.0027 - val_accuracy: 0.9995 - val_loss: 0.0027
Epoch 3/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9995 - val_loss: 0.0026
Epoch 4/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9995 - loss: 0.0021 - val_accuracy: 0.9995 - val_loss: 0.0024
Epoch 5/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9995 - loss: 0.0020 - val_accuracy: 0.9995 - val_loss: 0.0025
Epoch 6/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9996 - loss: 0.0018 - val_accuracy: 0.9994 - val_loss: 0.0025
Epoch 7/100
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9996 - loss: 0.0018 - val_accuracy: 0.9994 - val

In [7]:
csv_path = f'{OUTPUT_DIR}/results_fullprecision.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['distance', 'noise', 'decoder', 'p_L', 'model_size_kb'])
    writer.writeheader()
    writer.writerows(results)

print(f'\nResults saved to {csv_path}')
print(f'\nSummary:')
print('distance  noise   decoder      p_L  model_size_kb')
for r in results:
    print(f'{r["distance"]:8d}  {r["noise"]:6.3f}  {r["decoder"]:12s}  {r["p_L"]:8.6f}  {r["model_size_kb"]:10.1f}')

print(f'\nCompleted at {datetime.now()}')


Results saved to ./results/results_fullprecision.csv

Summary:
distance  noise   decoder      p_L  model_size_kb
       3   0.001  CNN-32bit     0.000470       165.9
       3   0.001  RCNN-32bit    0.000420       282.3
       3   0.005  CNN-32bit     0.008590       165.9
       3   0.005  RCNN-32bit    0.008500       282.3
       3   0.010  CNN-32bit     0.029300       165.9
       3   0.010  RCNN-32bit    0.029290       282.3
       3   0.050  CNN-32bit     0.308910       165.9
       3   0.050  RCNN-32bit    0.304990       282.3

Completed at 2026-05-29 01:32:46.871911
